# 05a - Configuration-Driven Pipelines: Basics

Nebula pipelines can be defined entirely in configuration (JSON/YAML), enabling:

- **Separation of concerns** - Data engineers define configs, code stays stable
- **Environment flexibility** - Different configs for dev/prod/regions
- **No-code changes** - Adjust pipelines without redeploying

| Part | Topic |
|------|-------|
| **1** | Basic Configuration |
| **2** | Transformer Options |

**Pipeline keywords covered in Part 2:**

| Keyword | Type | Description |
|---------|------|-------------|
| `{"store": "key"}` | dict | Save current DF to storage |
| `{"store_debug": "key"}` | dict | Save only when debug mode is on |
| `{"from_store": "key"}` | dict | Replace current DF from storage |
| `"to_lazy"` | string | Convert eager → lazy (Polars) |
| `"collect"` | string | Collect lazy → eager (Polars) |

In [1]:
import polars as pl

from nebula.base import Transformer
from nebula.pipelines.pipeline_loader import load_pipeline
from nebula.storage import nebula_storage as ns

In [2]:
orders = pl.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "customer": ["alice", "bob", "alice", "carol", "bob"],
    "amount": [150.0, 75.0, 200.0, 50.0, 300.0],
    "region": ["US", "EU", "US", "APAC", "EU"],
})
orders

order_id,customer,amount,region
i64,str,f64,str
1,"""alice""",150.0,"""US"""
2,"""bob""",75.0,"""EU"""
3,"""alice""",200.0,"""US"""
4,"""carol""",50.0,"""APAC"""
5,"""bob""",300.0,"""EU"""


---
## Part 1: Basic Configuration

The `load_pipeline` function accepts a Python dict (which can come from JSON or YAML files).

### 1.1 Minimal Pipeline

In [3]:
config = {
    "pipeline": [
        {"transformer": "SelectColumns", "params": {"columns": ["order_id", "amount"]}},
        {"transformer": "AssertNotEmpty"},
    ]
}

pipe = load_pipeline(config)
pipe.show(add_params=True)

*** Pipeline *** (2 transformations)
 - SelectColumns -> PARAMS: columns=['order_id', 'amount']
 - AssertNotEmpty


In [4]:
result = pipe.run(orders)
print(result)

2026-03-09 12:47:07,451 | [INFO]: Starting pipeline 
2026-03-09 12:47:07,452 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:47:07,460 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:47:07,460 | [INFO]: Running 'AssertNotEmpty' ... 
2026-03-09 12:47:07,465 | [INFO]: Completed 'AssertNotEmpty' in 0.0s 
2026-03-09 12:47:07,466 | [INFO]: Pipeline completed in 0.0s 


shape: (5, 2)
┌──────────┬────────┐
│ order_id ┆ amount │
│ ---      ┆ ---    │
│ i64      ┆ f64    │
╞══════════╪════════╡
│ 1        ┆ 150.0  │
│ 2        ┆ 75.0   │
│ 3        ┆ 200.0  │
│ 4        ┆ 50.0   │
│ 5        ┆ 300.0  │
└──────────┴────────┘


### 1.2 Loading from Files

In practice, configs come from files:

```python
import json
import yaml  # pip install pyyaml

# From JSON
with open("pipeline.json") as f:
    config = json.load(f)

# From YAML
with open("pipeline.yaml") as f:
    config = yaml.safe_load(f)

pipe = load_pipeline(config)
```

### 1.3 Pipeline-Level Options

In [5]:
config = {
    "name": "Order Processing",
    "df_input_name": "Raw Orders",
    "df_output_name": "Processed Orders",
    "pipeline": [
        {"transformer": "Filter", "params": {
            "input_col": "amount",
            "perform": "keep",
            "operator": "gt",
            "value": 100
        }},
    ]
}

pipe = load_pipeline(config)
pipe.show()

*** Order Processing *** (1 transformation)
 - Filter


---
## Part 2: Transformer Options

Each transformer entry supports these keys:

| Key | Required | Description |
|-----|----------|-------------|
| `transformer` | Yes | Transformer class name |
| `params` | No | Dict of constructor parameters |
| `description` | No | Human-readable description |
| `lazy` | No | If `true`, wrap in LazyWrapper |
| `skip` | No | If `true`, skip this transformer |
| `perform` | No | If `false`, skip this transformer |

In [6]:
config = {
    "pipeline": [
        {
            "transformer": "Filter",
            "description": "Keep only high-value orders",
            "params": {
                "input_col": "amount",
                "perform": "keep",
                "operator": "gt",
                "value": 100
            }
        },
        {
            "transformer": "DropColumns",
            "skip": True,  # This transformer is skipped
            "params": {"columns": "region"}
        },
        {
            "transformer": "SelectColumns",
            "perform": False,  # Same as skip: True
            "params": {"columns": ["order_id"]}
        },
    ]
}

pipe = load_pipeline(config)
pipe.show()

# Only Filter runs - DropColumns and SelectColumns are skipped
result = pipe.run(orders)
print(f"\nColumns: {result.columns}")

2026-03-09 12:47:07,495 | [INFO]: Starting pipeline 
2026-03-09 12:47:07,495 | [INFO]: Running 'Filter' ... 
2026-03-09 12:47:07,497 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:47:07,497 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (1 transformation)
 - Filter
     Description: Keep only high-value orders

Columns: ['order_id', 'customer', 'amount', 'region']


### 2.1 Feature Flags with Skip/Perform

Use environment variables or config flags to conditionally enable transformers:

In [7]:
# Simulate feature flags
ENABLE_EXPENSIVE_VALIDATION = False
ENABLE_DISCOUNT = True

config = {
    "pipeline": [
        {
            "transformer": "AssertCount",
            "description": "Expensive row count validation",
            "perform": ENABLE_EXPENSIVE_VALIDATION,
            "params": {"min_count": 1}
        },
        {
            "transformer": "AddLiterals",
            "description": "Add discount flag",
            "perform": ENABLE_DISCOUNT,
            "params": {
                "data": [{"alias": "has_discount", "value": True}]
            }
        },
    ]
}

pipe = load_pipeline(config)
pipe.show()

*** Pipeline *** (1 transformation)
 - AddLiterals
     Description: Add discount flag


### 2.2 Storage Keywords in Config

In [8]:
ns.clear()

config = {
    "pipeline": [
        {"transformer": "Filter", "params": {
            "input_col": "amount", "perform": "keep", "operator": "gt", "value": 100
        }},
        {"store": "filtered_orders"},
        {"transformer": "SelectColumns", "params": {"columns": ["order_id", "amount"]}},
    ]
}

pipe = load_pipeline(config)
pipe.show()

result = pipe.run(orders)
print(f"\nStored keys: {ns.list_keys()}")

2026-03-09 12:47:07,520 | [INFO]: Nebula Storage: clear. 
2026-03-09 12:47:07,520 | [INFO]: Nebula Storage: 0 keys remained after clearing. 
2026-03-09 12:47:07,520 | [INFO]: Starting pipeline 
2026-03-09 12:47:07,525 | [INFO]: Running 'Filter' ... 
2026-03-09 12:47:07,527 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:47:07,527 | [INFO]:    --> Store df with key "filtered_orders" 
2026-03-09 12:47:07,527 | [INFO]: Nebula Storage: setting an object (<class 'polars.dataframe.frame.DataFrame'>) with the key "filtered_orders". 
2026-03-09 12:47:07,527 | [INFO]: Running 'SelectColumns' ... 
2026-03-09 12:47:07,530 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:47:07,530 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline *** (2 transformations)
 - Filter
   --> Store df with key "filtered_orders"
 - SelectColumns

Stored keys: ['filtered_orders']


### 2.3 Eagerness Control Keywords

Two string keywords let you switch between Polars eager and lazy execution mid-pipeline.
Both are **Polars-only** — they raise a clear `TypeError` for pandas input.

| Keyword | Input | Output | No-op when… |
|---------|-------|--------|-------------|
| `"to_lazy"` | `pl.DataFrame` / `nw.DataFrame` | `pl.LazyFrame` / `nw.LazyFrame` | frame is already lazy |
| `"collect"` | `pl.LazyFrame` / `nw.LazyFrame` | `pl.DataFrame` / `nw.DataFrame` | frame is already eager |

**Why would you need this in a config-driven pipeline?**

- You receive an eager `DataFrame` but want Polars to optimize the following
  transformations as a single lazy query plan → add `"to_lazy"` first.
- A downstream transformer or export step requires an eager `DataFrame`
  (e.g., writing to Parquet with schema inference) → add `"collect"` before it.
- You need a checkpoint: `"collect"` materializes the result so you can
  inspect or store it, then `"to_lazy"` resumes lazy execution.

In [9]:
# "to_lazy": eager pl.DataFrame → lazy pl.LazyFrame
# orders is a pl.DataFrame — to_lazy defers computation to let
# Polars optimize all following steps as one query plan.

config = {"pipeline": ["to_lazy"]}

pipe = load_pipeline(config)
pipe.show()

result = pipe.run(orders)  # orders is eager
print(f"\nInput type : {type(orders)}")
print(f"Output type: {type(result)}")

2026-03-09 12:47:07,548 | [INFO]: Starting pipeline 
2026-03-09 12:47:07,548 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline ***
   --> Convert eager DataFrame to lazy frame (Polars only)

Input type : <class 'polars.dataframe.frame.DataFrame'>
Output type: <class 'polars.lazyframe.frame.LazyFrame'>


In [10]:
# "collect": lazy pl.LazyFrame → eager pl.DataFrame
# Materializes the query plan and returns a concrete DataFrame.

orders_lazy = orders.lazy()  # simulate receiving a lazy frame

config = {"pipeline": ["collect"]}

pipe = load_pipeline(config)
pipe.show()

result = pipe.run(orders_lazy)
print(f"\nInput type : {type(orders_lazy)}")
print(f"Output type: {type(result)}")
result

2026-03-09 12:47:07,565 | [INFO]: Starting pipeline 
2026-03-09 12:47:07,566 | [INFO]: Pipeline completed in 0.0s 


*** Pipeline ***
   --> Collect lazy frame into eager DataFrame (Polars only)

Input type : <class 'polars.lazyframe.frame.LazyFrame'>
Output type: <class 'polars.dataframe.frame.DataFrame'>


order_id,customer,amount,region
i64,str,f64,str
1,"""alice""",150.0,"""US"""
2,"""bob""",75.0,"""EU"""
3,"""alice""",200.0,"""US"""
4,"""carol""",50.0,"""APAC"""
5,"""bob""",300.0,"""EU"""


#### Composing `to_lazy` and `collect` in a real pipeline

A typical pattern: receive an eager `DataFrame`, switch to lazy so Polars can
optimize the filter + select as one pass, then `collect` at the end to hand an
eager result to the next stage.

```yaml
# pipeline.yaml — equivalent YAML
pipeline:
  - to_lazy
  - transformer: Filter
    params: {input_col: amount, perform: keep, operator: gt, value: 100}
  - transformer: SelectColumns
    params: {columns: [order_id, customer, amount]}
  - collect
```

In [11]:
# Realistic use-case: eager input → lazy optimization zone → eager output
# Polars fuses the Filter + SelectColumns into a single optimized query plan.

config = {
    "name": "Lazy-optimized order filter",
    "pipeline": [
        "to_lazy",  # ← switch to lazy: enables Polars query optimization
        {"transformer": "Filter", "params": {
            "input_col": "amount", "perform": "keep", "operator": "gt", "value": 100
        }},
        {"transformer": "SelectColumns", "params": {
            "columns": ["order_id", "customer", "amount"]
        }},
        "collect",  # ← materialize: downstream steps expect an eager DataFrame
    ]
}

pipe = load_pipeline(config)
pipe.show()

result = pipe.run(orders)  # feed an eager DataFrame
print(f"\nInput type : {type(orders)}")
print(f"Output type: {type(result)}")
result

2026-03-09 12:47:07,595 | [INFO]: Starting pipeline 'Lazy-optimized order filter' 
2026-03-09 12:47:07,595 | [INFO]: Running 'Filter' ... 
2026-03-09 12:47:07,595 | [INFO]: Completed 'Filter' in 0.0s 
2026-03-09 12:47:07,599 | [INFO]: Running 'SelectColumns' ... 
C:\Users\valer\Desktop\pypr\gitrepos\nebula\venv\lib\site-packages\narwhals\_polars\dataframe.py:152: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  return self.native.columns
2026-03-09 12:47:07,605 | [INFO]: Completed 'SelectColumns' in 0.0s 
2026-03-09 12:47:07,605 | [INFO]: Pipeline 'Lazy-optimized order filter' completed in 0.0s 


*** Lazy-optimized order filter *** (2 transformations)
   --> Convert eager DataFrame to lazy frame (Polars only)
 - Filter
 - SelectColumns
   --> Collect lazy frame into eager DataFrame (Polars only)

Input type : <class 'polars.dataframe.frame.DataFrame'>
Output type: <class 'polars.dataframe.frame.DataFrame'>


order_id,customer,amount
i64,str,f64
1,"""alice""",150.0
3,"""alice""",200.0
5,"""bob""",300.0


---
## Summary

In this notebook you learned:
- How to define pipelines as Python dicts (config format)
- Pipeline-level options (`name`, `split_function`, `branch`, `apply_to_rows`)
- Transformer options (`skip`, `perform`, `description`)
- Storage keywords and eagerness control

**Next notebook:** 05b covers custom transformers, lazy parameters, loops, and full examples in config.

In [12]:
ns.clear()

2026-03-09 12:47:07,637 | [INFO]: Nebula Storage: clear. 
2026-03-09 12:47:07,638 | [INFO]: Nebula Storage: 0 keys remained after clearing. 
